In [1]:
from pathlib import Path
from metasmith.agents import Agent
from metasmith.models.libraries import *
from metasmith.models.remote import *

from local.constants import WORKSPACE_ROOT
SOCKEYE_SOURCE = GlobusSource.Parse("https://app.globus.org/file-manager?origin_id=64a5c402-05c4-4607-bbad-46a9c2aebd98&origin_path=%2Fhome%2Ftxyliu%2F")
SOCKEYE_SOURCE.endpoint

'64a5c402-05c4-4607-bbad-46a9c2aebd98'

In [2]:
agent_local = Agent(
    home=Source.FromLocal(WORKSPACE_ROOT/"main/local_mock/cache/local_home"),
)

agent_ssh = Agent(
    home=SshSource(
        host="cosmos",
        path="~/workspace/metasmith_home",
    ).AsSource(),
)

agent_slurm = Agent(
    setup_commands=[
        "module load gcc/9.4.0 apptainer/1.3.1",
    ],
    home=SshSource(
        host="sockeye",
        path="~/scratch/metasmith_home",
    ).AsSource(),
    globus_uuid=SOCKEYE_SOURCE.endpoint,
)

# agent=agent_local
# agent=agent_ssh
agent=agent_slurm
agent.Deploy()

2025-03-14_13-35-07  | starting ssh to [sockeye]


2025-03-14_13-35-07 E| Pseudo-terminal will not be allocated because stdin is not a terminal.


2025-03-14_13-35-07  | ssh_connected_flag.1FQfCxG4
2025-03-14_13-35-08  | /scratch/st-shallam-1/pwy_group/metasmith_home
2025-03-14_13-35-08  | /arc/home/txyliu
2025-03-14_13-35-08  | >>> AGENT_HOME=/scratch/st-shallam-1/pwy_group/metasmith_home
2025-03-14_13-35-08  | >>> mkdir -p $AGENT_HOME
2025-03-14_13-35-08  | >>> mkdir -p /arc/home/txyliu/.globus
2025-03-14_13-35-08  | >>> mkdir -p /arc/home/txyliu/.globusonline
2025-03-14_13-35-08  | >>> [ -e /scratch/st-shallam-1/pwy_group/metasmith_home/metasmith.sif ] || apptainer pull /scratch/st-shallam-1/pwy_group/metasmith_home/metasmith.sif docker://quay.io/hallamlab/metasmith:latest
2025-03-14_13-35-08  | staged [msm_stub]
2025-03-14_13-35-08  | staged [msm]
2025-03-14_13-35-08  | >>> cd /scratch/st-shallam-1/pwy_group/metasmith_home && ./msm api deploy_from_container
2025-03-14_13-35-08  | including dev binds


2025-03-14_13-35-08 E| INFO:    squashfuse not found, will not be able to use gocryptfs
2025-03-14_13-35-08 E| INFO:    gocryptfs not found, will not be able to use gocryptfs


2025-03-14_13-35-09  | 2025-03-14_13-35-09  | api call to [deploy_from_container] with [{}]
2025-03-14_13-35-09  | 2025-03-14_13-35-09  | deploying to [/ws]
2025-03-14_13-35-09  | 2025-03-14_13-35-09  | deploying relay server to [/ws/relay/msm_relay]
2025-03-14_13-35-09  | 2025-03-14_13-35-09  | deployment complete
2025-03-14_13-35-10  | staged [lib/agent.yml]
2025-03-14_13-35-10  | staged [lib/msm_bootstrap]
2025-03-14_13-35-10  | staged [lib/nextflow_config]
2025-03-14_13-35-10  | deploying staged files
2025-03-14_13-35-12  | deployed to [ssh://sockeye:~/scratch/metasmith_home]


In [3]:
CACHE = WORKSPACE_ROOT/"main/local_mock/cache/xgdb_tests"
trlib = TransformInstanceLibrary.Load("./transforms/simple_1")
xgdb = DataInstanceLibrary.Load(CACHE/"test.xgdb")
# refdb = DataInstanceLibrary.Load(CACHE/"ref.xgdb")
types = DataTypeLibrary.Load(WORKSPACE_ROOT/"main/local_mock/prototypes/metagenomics.dev3.yml")

In [4]:
# chinook_ep = GlobusSource.Parse("https://app.globus.org/file-manager?origin_id=2602486c-1e0f-47a0-be15-eec1b0ff0f96&origin_path=%2FMetasmith%2F").endpoint
# refdb.SaveAs(GlobusSource(endpoint=chinook_ep, path="/Metasmith/ref.xgdb").AsSource())

refdb = DataInstanceLibrary.LoadFrom(
    src=GlobusSource.Parse("https://app.globus.org/file-manager?origin_id=2602486c-1e0f-47a0-be15-eec1b0ff0f96&origin_path=%2FMetasmith%2Fref.xgdb%2F").AsSource(),
    dest=CACHE/"ref.image.xgdb",
    as_image=True,
)
refdb.remote_src

Source(address='globus://2602486c-1e0f-47a0-be15-eec1b0ff0f96:/Metasmith/ref.xgdb', type=SourceType.GLOBUS)

In [5]:
task = agent.GenerateWorkflow(
    given=[xgdb, refdb],
    transforms=[trlib],
    targets=[
        types["orf_annotations"].WithLineage([
            types["contigs"],
            # xgdb["example.fna"].type,
        ]),
    ],
)

print(task.plan._key)
for step in task.plan.steps:
    print(step.transform.name)

aLR0oVmp
pprodigal
diamond


In [11]:
with open(WORKSPACE_ROOT/"secrets/slurm_account") as f:
    slurm_account = f.read().strip()
    
task.container_runtime = ContainerRuntime.APPTAINER
task.config = dict(
    nextflow = dict(
        preset = "slurm",
        slurm_account=slurm_account,
        # preset = "default",
    ),

)

In [7]:
agent.StageWorkflow(task, on_exist="clear")
# agent.StageWorkflow(task, on_exist="update")
# agent.StageWorkflow(task)

2025-03-14_13-35-21  | connecting to deployed agent
2025-03-14_13-35-21  | starting ssh to [sockeye]


 E| > Pseudo-terminal will not be allocated because stdin is not a terminal.


  | > ssh_connected_flag.1FQfCxG4
2025-03-14_13-35-22  | starting relay service


 E| > 2025-03-14_13-35-23 E| relay server already running in [relay/connections]


  | > 2025-03-14_13-35-23  | connecting to relay as [fQ7qWF3msTGS]
2025-03-14_13-35-24  | sending metadata for workflow [aLR0oVmp]
2025-03-14_13-35-30  | staging
  | > including dev binds


 E| > INFO:    squashfuse not found, will not be able to use gocryptfs
 E| > INFO:    gocryptfs not found, will not be able to use gocryptfs


  | > 2025-03-14_13-35-30  | api call to [stage_workflow] with [{'task_key': 'aLR0oVmp'}]
  | > 2025-03-14_13-35-31  | staging workflow [aLR0oVmp] with [2] data libs and [1] transform libs
  | > 2025-03-14_13-35-31  | ex| /scratch/st-shallam-1/pwy_group/metasmith_home
  | > 2025-03-14_13-35-31  | work [/ws/runs/aLR0oVmp]
  | > 2025-03-14_13-35-31  | data [/msm_home/data]
  | > 2025-03-14_13-35-31  | external work [/scratch/st-shallam-1/pwy_group/metasmith_home/runs/aLR0oVmp]
  | > 2025-03-14_13-35-31  | external data [/scratch/st-shallam-1/pwy_group/metasmith_home/data]
  | > 2025-03-14_13-35-31  | additional params:
  | > 2025-03-14_13-35-31  |     nextflow:
  | > 2025-03-14_13-35-31  |       preset: slurm
  | > 2025-03-14_13-35-31  | moving remote data libraries to [/msm_home/data]
  | > 2025-03-14_13-35-31  | using nextflow preset [slurm]
  | > 2025-03-14_13-35-32  | [aLR0oVmp] staged to [{AGENT_HOME}/runs/aLR0oVmp]
2025-03-14_13-35-32  | closing connection


In [8]:
# import shutil
# work_root = WORKSPACE_ROOT/"main/local_mock/cache/local_home/runs/kCvaS6w9"
# for p in [".nextflow", "nxf_logs", "nxf_work", "results"]:
#     shutil.rmtree(work_root/p, ignore_errors=True)
# shutil.rmtree(WORKSPACE_ROOT/"main/local_mock/mock/cache", ignore_errors=True)
    
agent.RunWorkflow(task)

2025-03-14_13-35-34  | connecting to deployed agent
2025-03-14_13-35-34  | starting ssh to [sockeye]


 E| > Pseudo-terminal will not be allocated because stdin is not a terminal.


  | > ssh_connected_flag.1FQfCxG4
2025-03-14_13-35-35  | starting relay service


 E| > 2025-03-14_13-35-35 E| relay server already running in [relay/connections]


  | > 2025-03-14_13-35-35  | connecting to relay as [J7JMTwH0eNfU]
2025-03-14_13-35-36  | executing workflow
  | > including dev binds


...

  | > 2025-03-14_13-36-33  |  N E X T F L O W   ~  version 24.10.5
  | > 2025-03-14_13-36-33  | 
  | > 2025-03-14_13-36-34  | WARN: It appears you have never run this project before -- Option `-resume` is ignored
  | > 2025-03-14_13-36-35  | Launching `./workflow.nf` [prickly_ampere] DSL2 - revision: 1b95ba5e23
  | > 2025-03-14_13-36-35  | 
  | > 2025-03-14_13-36-37  | [-        ] pprodigal__zgz5x1IB -
  | > 2025-03-14_13-36-37  | [-        ] diamond__lnNJuDqG   -
  | > 2025-03-14_13-36-37  | 
  | > 2025-03-14_13-36-37  | [-        ] pprodigal__zgz5x1IB | 0 of 1
  | > 2025-03-14_13-36-37  | [-        ] diamond__lnNJuDqG   -
  | > 2025-03-14_13-36-38  | 
  | > 2025-03-14_13-36-38  | [-        ] pprodigal__zgz5x1IB | 0 of 1
  | > 2025-03-14_13-36-38  | [-        ] diamond__lnNJuDqG   -
  | > 2025-03-14_13-36-38  | ERROR ~ Error executing process > 'pprodigal__zgz5x1IB (1)'
  | > 2025-03-14_13-36-38  | 
  | > 2025-03-14_13-36-38  | Caused by:
  | > 2025-03-14_13-36-38  |   Invalid SLURM s

 E| > Traceback (most recent call last):


  | > 2025-03-14_13-36-40  | compiling results


 E| >   File "/opt/conda/envs/metasmith_env/bin/metasmith", line 8, in <module>
 E| >     sys.exit(main())
 E| >              ^^^^^^
 E| >   File "/opt/conda/envs/metasmith_env/lib/python3.12/site-packages/metasmith/coms/cli.py", line 92, in main
 E| >     COMMANDS.get(# calls command function with args
 E| >   File "/opt/conda/envs/metasmith_env/lib/python3.12/site-packages/metasmith/coms/cli.py", line 67, in api
 E| >     HandleRequest(args.endpoint, body)
 E| >   File "/opt/conda/envs/metasmith_env/lib/python3.12/site-packages/metasmith/coms/api.py", line 37, in HandleRequest
 E| >     _ENDPOINTS[endpoint](api, body)
 E| >   File "/opt/conda/envs/metasmith_env/lib/python3.12/site-packages/metasmith/coms/api.py", line 27, in execute_workflow
 E| >     ExecuteWorkflow(key)
 E| >   File "/opt/conda/envs/metasmith_env/lib/python3.12/site-packages/metasmith/agents.py", line 536, in ExecuteWorkflow
 E| >     assert p.exists(), f"workflow failed to produce expected output [{x.dtype_name}] 

2025-03-14_13-36-41  | closing connection


In [9]:
# import mimetypes

# def istext(filename):
#     s=open(filename, encoding="latin1").read(512)
#     text_characters = "".join([chr(x) for x in range(32, 127)] + list("\n\r\t\b"))
#     translation_table = str.maketrans("", "", text_characters)
#     if not s:
#         # Empty files are considered text
#         return True
#     if "\0" in s:
#         # Files with null bytes are likely binary
#         return False
#     # Get the non-text characters (maps a character to itself then
#     # use the 'remove' option to get rid of the text characters.)
#     t = s.translate(translation_table)
#     # If more than 30% non-text characters, then
#     # this is considered a binary file
#     if float(len(t))/float(len(s)) > 0.30:
#         return False
#     return True

# # istext("/home/tony/workspace/tools/Metasmith/metasmith.sif")
# istext("/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/runs/dwfuH8Cz/nxf_work/dd/6d7e3eef979ec8010613bb376b63b6/container.diamond.oci.uri")

In [10]:
# from metasmith.coms.containers import ContainerRuntime

# s = ContainerRuntime.APPTAINER.name
# ContainerRuntime[s], s